In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras.preprocessing import image_dataset_from_directory
import pandas as pd
from tensorflow.keras.applications.efficientnet import preprocess_input

In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/EuroSAT.zip'
extract_path = '/content/eurosat'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
 # Makes results repeatable, so we don't get different accuracy each time.
def set_seed(seed=31415):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
set_seed(31415)

# Load CSVs containing the official train/valid/test splits
train_df = pd.read_csv('/content/eurosat/train.csv')
valid_df = pd.read_csv('/content/eurosat/validation.csv')
test_df  = pd.read_csv('/content/eurosat/test.csv')

BASE_PATH   = '/content/eurosat/'
IMAGE_SIZE  = (128, 128)
BATCH_SIZE  = 64
NUM_CLASSES = 10

# Build full image paths from the Filename column
train_df['full_path'] = BASE_PATH + train_df['Filename']
valid_df['full_path'] = BASE_PATH + valid_df['Filename']
test_df['full_path']  = BASE_PATH + test_df['Filename']

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE, method='bilinear')  # give new values to the new pixels
    image = preprocess_input(image)  # ← CRITICAL CHANGE                    # normalize pixels from 0-255 to 0-1
    return image, label

def make_dataset(df, shuffle=False):
    paths  = df['full_path'].values
    labels = tf.keras.utils.to_categorical(df['Label'].values, NUM_CLASSES)  # one-hot encode labels

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, seed=31415)               # randomizes training data to prevent overfitting
    ds = ds.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)
    return ds

ds_train = make_dataset(train_df, shuffle=True)
ds_valid = make_dataset(valid_df, shuffle=False)
ds_test  = make_dataset(test_df,  shuffle=False)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(128, 128, 3)
)

# include_top=False
# → removes original classifier (ImageNet 1000 classes)
# weights='imagenet'
# → uses pretrained knowledge (edges, shapes, textures)
# input_shape=(128,128,3)
# → matches your dataset

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,829 (16.08 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
model.compile(
    optimizer='adam', #update weights
    loss='categorical_crossentropy', #measures how well the the models predictions match the
                                     #true labels
    metrics=['accuracy']             #reports accuracy
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
small_ds = ds_train.take(1)

model.fit(small_ds, epochs=20)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 18s 18s/step - accuracy: 0.1094 - loss: 2.4505
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0938 - loss: 2.3847
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.1719 - loss: 2.2228
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.2656 - loss: 2.1150
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.1875 - loss: 2.1310
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3594 - loss: 1.8445
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.4531 - loss: 1.7025
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5625 - loss: 1.5278
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5781 - loss: 1.5428
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.6719 - loss: 1.3334
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5625 - loss: 1.3841
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.7188 - loss: 1.1430
Epoch 13/20
1/1 ━━━━━━━

In [ ]:
history = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 608s 2s/step - accuracy: 0.8735 - loss: 0.3898 - val_accuracy: 0.9293 - val_loss: 0.2128
Epoch 2/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 596s 2s/step - accuracy: 0.9109 - loss: 0.2652 - val_accuracy: 0.9400 - val_loss: 0.1837
Epoch 3/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 630s 2s/step - accuracy: 0.9185 - loss: 0.2369 - val_accuracy: 0.9383 - val_loss: 0.1803
Epoch 4/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 615s 2s/step - accuracy: 0.9279 - loss: 0.2164 - val_accuracy: 0.9381 - val_loss: 0.1782
Epoch 5/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 587s 2s/step - accuracy: 0.9300 - loss: 0.2007 - val_accuracy: 0.9444 - val_loss: 0.1701
Epoch 6/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 606s 2s/step - accuracy: 0.9352 - loss: 0.1861 - val_accuracy: 0.9439 - val_loss: 0.1700
Epoch 7/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 595s 2s/step - accuracy: 0.9374 - loss: 0.1836 - val_accuracy: 0.9452 - val_loss: 0.1657
Epoch 8/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 580s 2s/step - accuracy: 0.9408 - loss: 0.1715 - val_accu

In [ ]:
test_loss, test_accuracy = model.evaluate(ds_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

43/43 ━━━━━━━━━━━━━━━━━━━━ 65s 2s/step - accuracy: 0.9544 - loss: 0.1445
Test Accuracy: 0.9544


In [ ]:
#PLOT TRAINING HISTORY
plt.figure(figsize=(12, 4))  # Creates a figure 12 inches wide, 4 inches tall

# First subplot (accuracy graph)
plt.subplot(1, 2, 1)  # 1 row, 2 columns, this is plot #1
plt.plot(history.history['accuracy'], label='Train Accuracy')      # Blue line
plt.plot(history.history['val_accuracy'], label='Val Accuracy')    # Orange line
plt.title('Accuracy over epochs')
plt.legend()  # Shows which line is which

# Second subplot (loss graph)
plt.subplot(1, 2, 2)  # 1 row, 2 columns, this is plot #2
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over epochs')
plt.legend()

plt.show()  # Display both graphs

NameError: name 'plt' is not defined

In [ ]:
model.save("/content/drive/MyDrive/AI_OLYMPICS/Project/final_model.keras")

NameError: name 'model' is not defined

In [ ]:
model.save('/content/drive/MyDrive/AI_OLYMPICS/Project/final_model.h5', save_format='h5')

NameError: name 'model' is not defined